## Quick analysis of podcast reviews

This notebook loads podcast reviews from a CSV file (like the ones produced by the Apple Store scraping notebook) and runs a few quick analyses of the star ratings. You'll see a preview of the data in table format, followed by several different views of the same `rating` column — each highlighting something the others don't.

**Note:** This notebook runs via Live Code, directly in your browser. Upload your own CSV file below — nothing is saved automatically, and each person works with their own file for this session only.

## Import modules

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

## Upload your data

Click **Upload** and choose a reviews CSV file from your own computer (tab-separated, as produced by the scraping notebook).

In [ ]:
import io

df = None

upload_widget = widgets.FileUpload(accept='.csv', multiple=False, description='Upload CSV')
upload_output = widgets.Output()

def _handle_upload(change):
    global df
    with upload_output:
        clear_output()
        if not upload_widget.value:
            print("No file selected yet.")
            return
        value = upload_widget.value
        if isinstance(value, dict):
            item = next(iter(value.values()))
        else:
            item = value[0]
        raw_bytes = bytes(item['content'])
        df = pd.read_csv(io.BytesIO(raw_bytes), sep='\t')
        print(f"Loaded {len(df)} rows.")

upload_widget.observe(_handle_upload, names='value')
display(upload_widget, upload_output)

## Preview the data

In [ ]:
if df is None:
    print("No data loaded yet — upload a CSV file above first.")
else:
    display(df)

## Prepare the data

Parse the review dates and extract the year and month, so we can group reviews by time period.

In [ ]:
if df is not None:
    df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d %H:%M:%S')
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year

    dfmonthly_median = df.groupby(["year", "month"]).median(numeric_only=True)
    dfmonthly_mean = df.groupby(["year", "month"]).mean(numeric_only=True)
    dfmonthly_count = df.groupby(["year", "month"]).size()

    print("Data prepared!")
else:
    print("No data loaded yet — upload a CSV file above first.")

## Visualisation 1: Median rating per month

The original view — how the typical (median) rating moved over time. Medians are robust to a handful of extreme reviews, but they can also hide how spread out opinions really were in a given month.

In [ ]:
if df is not None:
    dfmonthly_median['rating'].plot(title='Median rating per month', ylabel='Median rating', xlabel='Year, month')
    plt.show()

## Visualisation 2: Mean vs. median rating per month

Plotting the mean alongside the median shows where a small number of very high or very low ratings are pulling the average away from what most reviewers actually said. If the two lines track closely, ratings in that month were fairly consistent; where they diverge, a few outlier reviews are having an outsized effect on the mean.

In [ ]:
if df is not None:
    fig, ax = plt.subplots()
    dfmonthly_mean['rating'].plot(ax=ax, label='Mean', title='Mean vs. median rating per month')
    dfmonthly_median['rating'].plot(ax=ax, label='Median')
    ax.set_ylabel('Rating')
    ax.set_xlabel('Year, month')
    ax.legend()
    plt.show()

## Visualisation 3: Distribution of star ratings

App Store reviews are often bimodal — lots of 1-star and 5-star reviews, with fewer in between — which a single average or median completely hides. This chart shows how many reviews fall into each star rating overall.

In [ ]:
if df is not None:
    df['rating'].value_counts().sort_index().plot(
        kind='bar', title='Distribution of star ratings', xlabel='Star rating', ylabel='Number of reviews'
    )
    plt.show()

## Visualisation 4: Review volume per month

A rating trend is easier to interpret once you know how many reviews it's based on. A month with an average of 4.8 from 300 reviews means something quite different from a month with an average of 4.8 from 2 reviews.

In [ ]:
if df is not None:
    dfmonthly_count.plot(
        kind='bar', title='Number of reviews per month', xlabel='Year, month', ylabel='Number of reviews'
    )
    plt.show()

## Visualisation 5: Rating spread per year

A box plot shows the full spread of ratings within each year — the middle 50% (the box), the median (the line inside it), and outliers — rather than collapsing everything into a single number.

In [ ]:
if df is not None:
    df.boxplot(column='rating', by='year')
    plt.title('Rating spread per year')
    plt.suptitle('')
    plt.xlabel('Year')
    plt.ylabel('Rating')
    plt.show()